In [ ]:
import polars as pl

df = pl.read_csv("../data/processed/gbd_2023_death_rates.csv")
df.group_by("precision").agg(
    rows=pl.len(),
    zeros=(pl.col("rate_per_100k") == 0).sum(),
)

In [ ]:
road = (
    df.filter(cause_name="Road injuries", age_group="Age-standardized")
    .with_columns(rank=pl.col("rate_per_100k").rank("min", descending=True).over("sex"))
    .sort("sex", "rank")
    .select("sex", "rank", "location_name", "rate_per_100k", "lower", "upper")
)
road.filter(sex="Male").head(10)

In [ ]:
import plotly.graph_objects as go

# One row per country: male and female age-standardized rate side by side
wide = (
    df.filter(cause_name="Road injuries", age_group="Age-standardized")
    .pivot(on="sex", index="location_name", values="rate_per_100k")
    .with_columns(mean=(pl.col("Male") + pl.col("Female")) / 2)  # for sorting only
    .sort("mean")  # ascending -> highest ends up at the top in plotly
)


countries = wide["location_name"].to_list()

# Connector lines: x pairs separated by None so plotly breaks the line between rows
line_x, line_y = [], []
for c, f, m in wide.select("location_name", "Female", "Male").iter_rows():
    line_x += [f, m, None]
    line_y += [c, c, None]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=line_x,
        y=line_y,
        mode="lines",
        line=dict(color="#c3c2b7", width=2),
        hoverinfo="skip",
        showlegend=False,
    )
)
for sex, color in [("Female", "#eb6834"), ("Male", "#2a78d6")]:
    fig.add_trace(
        go.Scatter(
            x=wide[sex].to_list(),
            y=countries,
            mode="markers",
            name=sex,
            marker=dict(color=color, size=8, line=dict(color="white", width=2)),
            hovertemplate="%{y}<br>" + sex + ": %{x:.1f} per 100k<extra></extra>",
        )
    )

fig.update_layout(
    title="Road injury deaths per 100,000, age-standardized, 2023 (GBD)",
    height=len(countries) * 16 + 150,  # ~16px per country -> ~3,400px tall
    xaxis=dict(title="Deaths per 100,000", side="top", gridcolor="#eeeeee"),
    yaxis=dict(tickfont=dict(size=10), gridcolor="#f5f5f5"),
    legend=dict(orientation="h", y=1.02, x=0),
    plot_bgcolor="white",
    margin=dict(l=180),
)
fig.show()